# Check Missing Yelp Summaries
This notebook identifies restaurants where yelp_summary was not generated

In [1]:
import json
import pandas as pd

In [2]:
# Load the data
with open('restaurant_yelp_summarization.json', 'r', encoding='utf-8') as f:
    restaurants = json.load(f)

print(f"Total restaurants: {len(restaurants)}")

Total restaurants: 628


In [3]:
# Separate into categories
with_summary = []
without_summary = []
no_field = []

for r in restaurants:
    if 'yelp_summary' not in r:
        no_field.append(r)
    elif r.get('yelp_summary') is None:
        without_summary.append(r)
    else:
        with_summary.append(r)

print(f"With summary: {len(with_summary)}")
print(f"Summary is null (skipped): {len(without_summary)}")
print(f"No yelp_summary field (not processed): {len(no_field)}")

With summary: 617
Summary is null (skipped): 11
No yelp_summary field (not processed): 0


## Restaurants with null summaries (were skipped)

In [4]:
# Create dataframe of skipped restaurants with reasons
skipped_data = []
for r in without_summary:
    has_categories = bool(r.get('yelp_categories'))
    highlights = r.get('yelp_highlights', {}) or {}
    review_highlights = highlights.get('review_highlights', [])
    yelp_matched = r.get('yelp_matched', False)
    
    # Count highlights > 5%
    highlights_gt_5 = 0
    if review_highlights:
        for h in review_highlights:
            percent_str = h.get('review_count_percent', '0%')
            try:
                percent = float(percent_str.rstrip('%'))
                if percent > 5.0:
                    highlights_gt_5 += 1
            except:
                pass
    
    reason = "No categories" if not has_categories else f"No highlights > 5% (has {len(review_highlights)} total highlights)"
    
    skipped_data.append({
        'name': r.get('name', 'Unknown'),
        'borough': r.get('borough', ''),
        'neighborhood': r.get('neighborhood', ''),
        'yelp_matched': yelp_matched,
        'has_categories': has_categories,
        'total_highlights': len(review_highlights),
        'highlights_gt_5_pct': highlights_gt_5,
        'reason': reason
    })

df_skipped = pd.DataFrame(skipped_data)
print(f"\nRestaurants with null summaries (skipped): {len(df_skipped)}")
df_skipped


Restaurants with null summaries (skipped): 11


,name,borough,neighborhood,yelp_matched,has_categories,total_highlights,highlights_gt_5_pct,reason
0,Palermo Argentinian Bistro - Gramercy,Manhattan,Gramercy,False,False,0,0,No categories
1,Hav & Mar,Manhattan,Chelsea,False,False,0,0,No categories
2,Evalyn’s Tap House,Brooklyn,Boerum Hill,False,False,0,0,No categories
3,Estiatorio Milos Hudson Yards,Manhattan,Hudson Yards,False,False,0,0,No categories
4,Marseille,Manhattan,Hells Kitchen,True,False,0,0,No categories
5,Dowling’s at The Carlyle,Manhattan,Upper East Side,False,False,0,0,No categories
6,Margaux by La Sirène,Manhattan,Murray Hill,False,False,0,0,No categories
7,Akoya,Manhattan,Times Square/Theatre District,False,False,0,0,No categories
8,Little Fino,Brooklyn,Williamsburg,False,False,0,0,No categories
9,Cafe Zaffri,Manhattan,Union Square,False,False,0,0,No categories
